# GEAK-OpenEvolve: LLM-Guided GPU Kernel Optimization

This hands-on tutorial demonstrates how to use **GEAK-OpenEvolve** to automatically optimize GPU kernels using Large Language Models (LLMs).

##  What is OpenEvolve?

OpenEvolve uses an evolutionary algorithm powered by LLMs to iteratively improve GPU kernel performance. It:
1. Takes an initial kernel as a starting point
2. Uses an LLM to generate optimized variants
3. Evaluates each variant for correctness and performance
4. Evolves the population toward better solutions

## Prerequisites

Before running this notebook, you need:
- **OpenAI API Key**: Set in Step 2 below (required for LLM calls)
- **AMD GPU**: For running and benchmarking kernels

## 📋 Tutorial Steps

| Step | Description |
|------|-------------|
| 1 | Environment Setup |
| 2 | Set API Key |
| 3 | Verify Installation |
| 4 | Select Initial Kernel |
| 5 | Setup Evaluator |
| 6 | Configure Evolution |
| 7 | Pre-flight Validation |
| 8 | Run Evolution |

## 📁 Output Location

Results will be saved to: `GEAK-openevolve/tutorial/runs/`

In [ ]:
# Step 1: Environment Setup
import os
import sys
from pathlib import Path

# Get geak-openevolve root (GEAK-openevolve cloned from geak-openevolve branch)
# In Docker: /app/jupyter/src/geak-evolve/GEAK-openevolve
# Locally: Path.cwd() / "GEAK-openevolve" or parent directory
OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    # Fallback for local development
    OPENEVOLVE_ROOT = Path.cwd().parent
print(f"OpenEvolve Root: {OPENEVOLVE_ROOT}")

# Add to Python path
if str(OPENEVOLVE_ROOT) not in sys.path:
    sys.path.insert(0, str(OPENEVOLVE_ROOT))

print(f"\n✅ OpenEvolve root: {OPENEVOLVE_ROOT}")
print(f"✅ Python path updated")


In [ ]:
# Step 1.5: Verify GEAK-eval (pre-installed in Docker)
import os
from pathlib import Path
import subprocess

# Use same OPENEVOLVE_ROOT logic
OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent
GEAK_EVAL_DIR = OPENEVOLVE_ROOT / "GEAK-eval-OE"

# Verify GEAK-eval exists (should be pre-cloned in Docker)
if GEAK_EVAL_DIR.exists():
    print(f"✅ GEAK-eval exists at: {GEAK_EVAL_DIR}")
    
    # Check if geak-eval command is available
    result = subprocess.run(["which", "geak-eval"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ geak-eval command available: {result.stdout.strip()}")
    else:
        print("✅ GEAK-eval directory found (command may not be in PATH)")
else:
    print(f"❌ GEAK-eval not found at: {GEAK_EVAL_DIR}")
    print("   Please ensure you're running in the Docker container or clone manually:")


In [ ]:
# Step 2: Set Environment Variables
import os
from pathlib import Path

# ⚠️ IMPORTANT: Set your OpenAI API key here!
# Uncomment the line below and add your key:
# os.environ['OPENAI_API_KEY'] = "sk-your-api-key-here"

# Set ROCM_GOLDEN_DATA_PATH
OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent
GOLDEN_DATA_PATH = OPENEVOLVE_ROOT / "GEAK-eval-OE/geak_eval/data/ROCm/data/performance/golden_results"
os.environ['ROCM_GOLDEN_DATA_PATH'] = str(GOLDEN_DATA_PATH)

# Check API key status
api_key = os.environ.get('OPENAI_API_KEY')
if api_key:
    print(f"✅ OPENAI_API_KEY is set (length: {len(api_key)} chars)")
else:
    print("❌ OPENAI_API_KEY is NOT set!")
    print("   Please uncomment the line above and add your OpenAI API key")

print(f"✅ ROCM_GOLDEN_DATA_PATH = {GOLDEN_DATA_PATH}")
print(f"   Path exists: {GOLDEN_DATA_PATH.exists()}")


In [ ]:
# Step 3: Verify OpenEvolve Installation
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

try:
    import triton
    print(f"Triton: {triton.__version__}")
except:
    print("❌ Triton not found")

try:
    import openevolve
    print(f"OpenEvolve: {openevolve.__version__ if hasattr(openevolve, '__version__') else 'installed'}")
except:
    print("❌ OpenEvolve not found - install with: pip install -e .")

print("\n✅ Environment ready!")


---

##  Kernel Preparation

OpenEvolve requires three components to run:

| Component | Description |
|-----------|-------------|
| **Initial Kernel** | The starting Triton kernel code to optimize |
| **Evaluator** | Measures correctness and performance of each kernel variant |
| **Configuration** | Evolution parameters (iterations, population size, LLM model, etc.) |

### Example Kernel

We'll optimize a simple **vector addition kernel** (`add_kernel`) as a demonstration. This kernel:
- Adds two input tensors element-wise
- Uses Triton's block-based programming model
- Serves as a good starting point for understanding the optimization process

The LLM will attempt to improve this kernel by exploring different:
- Block sizes and tiling strategies
- Memory access patterns
- Triton-specific optimizations


In [ ]:
# Step 4: Select Example Kernel
from pathlib import Path

OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent
TUTORIAL_DIR = OPENEVOLVE_ROOT / "tutorial"

# Use kernel from GEAK-eval-OE (cloned GEAK-eval repository)
INITIAL_KERNEL = OPENEVOLVE_ROOT / "GEAK-eval-OE/geak_eval/data/ROCm/data/ROCm_v1/test_add_kernel.py"

if INITIAL_KERNEL.exists():
    print(f"✅ Selected kernel: {INITIAL_KERNEL.name}")
    print(f"   Path: {INITIAL_KERNEL.relative_to(OPENEVOLVE_ROOT)}")
    
    # Quick peek at the kernel
    with open(INITIAL_KERNEL, 'r') as f:
        lines = f.readlines()
    
    # Find the kernel function
    in_kernel = False
    kernel_lines = []
    for line in lines:
        if '@triton.jit' in line:
            in_kernel = True
        if in_kernel:
            kernel_lines.append(line.rstrip())
            if line.strip().startswith('tl.store') and 'output' in line:
                break
    
    print(f"\n📝 Kernel Preview:")
    for line in kernel_lines[:15]:
        print(f"   {line}")
    if len(kernel_lines) > 15:
        print(f"   ... ({len(kernel_lines)-15} more lines)")
else:
    print(f"❌ Kernel not found at: {INITIAL_KERNEL}")
    INITIAL_KERNEL = None


In [ ]:
# Step 5: Setup Evaluator
from pathlib import Path

OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent

# Use the ROCm evaluator from examples
EVALUATOR_PATH = OPENEVOLVE_ROOT / "examples/tb/rocm_evaluator.py"

if EVALUATOR_PATH.exists():
    print(f"✅ Using evaluator: {EVALUATOR_PATH.name}")
    print(f"   Path: {EVALUATOR_PATH.relative_to(OPENEVOLVE_ROOT)}")
else:
    print(f"❌ Evaluator not found at: {EVALUATOR_PATH}")
    EVALUATOR_PATH = None


In [ ]:
# Step 6: Configure Evolution Parameters
import yaml
import os
from pathlib import Path

OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent
TUTORIAL_DIR = OPENEVOLVE_ROOT / "tutorial"

# Configuration parameters - EASILY ADJUSTABLE
MAX_ITERATIONS = 3
POPULATION_SIZE = 50
NUM_ISLANDS = 4
LOG_LEVEL = "WARNING"

# LLM Configuration - Use OpenAI
LLM_MODEL = "gpt-4o"  # Options: gpt-4o, gpt-4-turbo, gpt-3.5-turbo
LLM_API_BASE = "https://api.openai.com/v1"  # Standard OpenAI endpoint

# Try multiple config templates
CONFIG_TEMPLATES = [
    OPENEVOLVE_ROOT / "configs/default_config.yaml",
    OPENEVOLVE_ROOT / "examples/tb/configs/demo_config.yaml",
]

CONFIG_FILE = TUTORIAL_DIR / "tutorial_config.yaml"

# Find first available template
template_found = None
for template in CONFIG_TEMPLATES:
    if template.exists():
        template_found = template
        print(f"✅ Found config template: {template.relative_to(OPENEVOLVE_ROOT)}")
        break

if template_found:
    with open(template_found, 'r') as f:
        config = yaml.safe_load(f)
    
    config['max_iterations'] = MAX_ITERATIONS
    config['log_level'] = LOG_LEVEL
    
    if 'database' not in config:
        config['database'] = {}
    config['database']['population_size'] = POPULATION_SIZE
    config['database']['num_islands'] = NUM_ISLANDS
    config['database']['log_prompts'] = True
    
    # CRITICAL: Fix db_path (can't be None)
    if config['database'].get('db_path') is None:
        config['database']['db_path'] = 'program_database'
    
    if 'llm' not in config:
        config['llm'] = {}
    
    # CRITICAL: Set sampling configuration
    config['llm']['sampling'] = {'fn': 'random'}
    
    # Use OpenAI API
    config['llm']['models'] = [{'name': LLM_MODEL, 'weight': 1.0}]
    config['llm']['evaluator_models'] = [{'name': LLM_MODEL, 'weight': 1.0}]
    config['llm']['api_base'] = LLM_API_BASE
    config['llm']['api_key'] = os.environ.get('OPENAI_API_KEY')  # Use env variable
    
    if 'evaluator' not in config:
        config['evaluator'] = {}
    config['evaluator']['cascade_evaluation'] = False
    config['evaluator']['verbose'] = False
    
    config['diff_based_evolution'] = True
    config['max_code_length'] = 50000
    
    # CRITICAL: Create evals directory for evaluator temp files
    evals_dir = TUTORIAL_DIR / "evals"
    evals_dir.mkdir(exist_ok=True)
    
    with open(CONFIG_FILE, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    print(f"✅ Configuration saved to: {CONFIG_FILE.name}")
    print(f"\n📝 Evolution Parameters:")
    print(f"  Max Iterations:  {MAX_ITERATIONS}")
    print(f"  Population Size: {POPULATION_SIZE}")
    print(f"  Num Islands:     {NUM_ISLANDS}")
    print(f"  Log Level:       {LOG_LEVEL}")
    print(f"  LLM Model:       {LLM_MODEL}")
    print(f"  LLM API Base:    {LLM_API_BASE}")
    print(f"  LLM Sampling:    random")
    print(f"  Database Path:   {config['database']['db_path']}")
    print(f"  API Key Set:     {'✅ Yes' if os.environ.get('OPENAI_API_KEY') else '❌ No - set OPENAI_API_KEY!'}")
    print(f"\n✅ Ready to run evolution!")
else:
    print("❌ No config template found!")
    CONFIG_FILE = None


In [ ]:
# Step 7: Setup Output Directory and Validate
from pathlib import Path
from datetime import datetime

OPENEVOLVE_ROOT = Path.cwd() / "GEAK-openevolve"
if not OPENEVOLVE_ROOT.exists():
    OPENEVOLVE_ROOT = Path.cwd().parent
TUTORIAL_DIR = OPENEVOLVE_ROOT / "tutorial"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = TUTORIAL_DIR / "runs" / f"tutorial_run_{timestamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CRITICAL: Ensure evals directory exists (needed by evaluator)
EVALS_DIR = TUTORIAL_DIR / "evals"
EVALS_DIR.mkdir(exist_ok=True)

print(f"✅ Output directory: {OUTPUT_DIR.relative_to(TUTORIAL_DIR)}")
print(f"✅ Evals directory: {EVALS_DIR.relative_to(TUTORIAL_DIR)}")

print("\n" + "="*70)
print("📋 Pre-Flight Check")
print("="*70)

try:
    kernel_var = INITIAL_KERNEL
    kernel_defined = True
except NameError:
    kernel_var = None
    kernel_defined = False

try:
    evaluator_var = EVALUATOR_PATH
    evaluator_defined = True
except NameError:
    evaluator_var = None
    evaluator_defined = False

try:
    config_var = CONFIG_FILE
    config_defined = True
except NameError:
    config_var = None
    config_defined = False

components = {
    "Kernel": (kernel_var, kernel_defined),
    "Evaluator": (evaluator_var, evaluator_defined),
    "Config": (config_var, config_defined)
}

all_ready = True
missing_cells = []

for name, (path, is_defined) in components.items():
    if not is_defined:
        print(f"❌ {name:12s}: NOT DEFINED (run earlier cell)")
        all_ready = False
        if name == "Kernel":
            missing_cells.append("Cell 5")
        elif name == "Evaluator":
            missing_cells.append("Cell 6")
        elif name == "Config":
            missing_cells.append("Cell 7")
    elif path and Path(path).exists():
        print(f"✅ {name:12s}: {Path(path).name}")
    else:
        print(f"❌ {name:12s}: NOT FOUND")
        all_ready = False

print("="*70)

if all_ready:
    print("\n🚀 All components ready! You can proceed to run evolution.")
else:
    print("\n⚠️  Some components are missing!")
    if missing_cells:
        print("\n📝 Please run these cells first:")
        for cell in missing_cells:
            print(f"   • {cell}")


In [ ]:
# Step 8: Run OpenEvolve Evolution
import subprocess
import os
from pathlib import Path

if not (INITIAL_KERNEL and EVALUATOR_PATH and CONFIG_FILE):
    print("❌ Missing required components!")
    print(f"   Kernel:    {INITIAL_KERNEL is not None and Path(INITIAL_KERNEL).exists()}")
    print(f"   Evaluator: {EVALUATOR_PATH is not None and Path(EVALUATOR_PATH).exists()}")
    print(f"   Config:    {CONFIG_FILE is not None and Path(CONFIG_FILE).exists()}")
else:
    command = [
        "openevolve-run",
        str(INITIAL_KERNEL),
        str(EVALUATOR_PATH),
        "--config", str(CONFIG_FILE),
        "--output", str(OUTPUT_DIR)
    ]
    
    print("🚀 Starting OpenEvolve Evolution...")
    print("="*70)
    print(f"📦 Kernel:    {Path(INITIAL_KERNEL).name}")
    print(f"⚙️  Evaluator: {Path(EVALUATOR_PATH).name}")
    print(f"📋 Config:    {Path(CONFIG_FILE).name}")
    print(f"📁 Output:    {OUTPUT_DIR.relative_to(TUTORIAL_DIR)}")
    print(f"🏠 Working Dir: {TUTORIAL_DIR}")
    print("="*70)
    print(f"\n$ cd {TUTORIAL_DIR}")
    print(f"$ {' '.join(command)}\n")
    print("="*70)
    
    # CRITICAL: Run from tutorial directory where evals/ exists
    result = subprocess.run(
        command, 
        capture_output=False, 
        text=True,
        cwd=str(TUTORIAL_DIR)  # Run from tutorial directory
    )
    
    print("="*70)
    if result.returncode == 0:
        print("\n✅ Evolution completed successfully!")
        print(f"\n📊 Results saved to: {OUTPUT_DIR.relative_to(TUTORIAL_DIR)}")
    else:
        print(f"\n❌ Evolution failed with exit code: {result.returncode}")


---

##  Understanding the Results

After evolution completes, your results are saved in:
```
GEAK-openevolve/tutorial/runs/tutorial_run_YYYYMMDD_HHMMSS/
```

### Output Structure

| Folder | Contents |
|--------|----------|
| `best/` | Best optimized kernel found (`best_program.py`) and performance info |
| `logs/` | Evolution logs showing progress across iterations |
| `program_database/` | All kernel variants tried during evolution |

### What to Look For

1. **Speedup**: Compare the final kernel's latency vs. baseline
2. **Correctness**: All variants are tested for correctness before being accepted
3. **Evolution Progress**: Check logs to see how performance improved over iterations

### Next Steps

- Try increasing `MAX_ITERATIONS` for more optimization opportunities
- Experiment with different initial kernels (e.g., matmul, softmax)
- Adjust `POPULATION_SIZE` and `NUM_ISLANDS` for different search strategies
